In [1]:
import csv
import json
import requests
import pandas as pd
import numpy as np



In [2]:
# age, fee_inr and prior_no_shows to int
def parse_row(row):

    error_list = []
    has_error = False
    try:
        age = int(row.get('age'))
        row.update({
            'age': age
        })
    except ValueError:
        has_error = True
        error_list.append('Invalid Age')

    try:
        fee_inr = int(row.get('fee_inr'))
        row.update({
            'fee_inr': fee_inr
        })
    except ValueError:
        error_list.append('Invalid Fee INR')
        has_error = True

    try:
        prior_no_shows = int(row.get('prior_no_shows'))
        row.update({
            'prior_no_shows': prior_no_shows
        })
    except ValueError:
        error_list.append('Invalid Prior No Shows')
        has_error = True

    try:
        distance_km = float(row.get('distance_km'))
        row.update({
            'distance_km': distance_km
        })
    except ValueError:
        error_list.append('Invalid Distance')
        has_error = True

    return {
        'data': 'row',
        'has_error': has_error,
        'errors': ','.join(e for e in error_list)
    }

In [3]:
class DataValidationPipeline:

    def __init__(self, schema):
        self.schema = schema
        self.errors = []

    def validate_record(self, record, idx):
        for field, rules in self.schema.items():
            value = record.get(field)
            if rules.get("required") and value is None:
                self.errors.append(f"Row {idx}: {field} is required")
                continue
            if value is not None and "type" in rules:
                if not isinstance(value, rules["type"]):
                    self.errors.append(f"Row {idx}: {field} wrong type")
        return len(self.errors) == 0

    def run(self, records):
        valid = []
        for i, record in enumerate(records):
            if self.validate_record(record, i):
                valid.append(record)
                print(
                    f"Valid: {len(valid)}/{len(records)}, "
                    f"Errors: {len(self.errors)}")

        return valid

In [4]:
def generate_json_file(data):

    with open('../data/rejected_data.json', 'w') as f:
        json.dump(data, f, indent=2)

In [5]:
file_path = "../data/qDCCmpyfss1kBJJB.csv"

In [6]:
appointment_data = list()
rejected_data = list()
with open(file_path, 'r') as file:
    csv_reader = csv.DictReader(file)
    print(csv_reader.fieldnames)
    row_count = 0
    for row in csv_reader:
        row_count += 1
        parsed_row = parse_row(row)
        if parsed_row.get('has_error'):
            row.update({
                'errors': parsed_row.get('errors')
            })
            rejected_data.append(row)
            continue
        appointment_data.append(parsed_row.get('data'))
    print(row_count)
    print(rejected_data)


generate_json_file(rejected_data)

['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
1020
[{'appointment_id': 'A0050', 'patient_id': 'P117', 'clinic_id': 'C04', 'booked_on': '2025-01-16', 'appt_date': '2025-01-24', 'age': 41, 'gender': 'M', 'distance_km': '', 'fee_inr': 600, 'booking_channel': 'Walk-in', 'visit_type': 'Consultation', 'reminder_sent': 'No', 'prior_no_shows': 0, 'status': 'Attended', 'errors': 'Invalid Distance'}, {'appointment_id': 'A0060', 'patient_id': 'P406', 'clinic_id': 'C02', 'booked_on': '2025-05-02', 'appt_date': '2025-05-11', 'age': '', 'gender': 'F', 'distance_km': 8.3, 'fee_inr': 700, 'booking_channel': 'App', 'visit_type': 'Follow-up', 'reminder_sent': 'Yes', 'prior_no_shows': 1, 'status': 'Attended', 'errors': 'Invalid Age'}, {'appointment_id': 'A0105', 'patient_id': 'P487', 'clinic_id': 'C08', 'booked_on': '2025-03-27', 'appt_date': '2025-05-11', 'ag

In [7]:
df = pd.read_csv(file_path)
df.shape
df.info()
df.isnull().sum()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB


np.int64(20)

 (a) drop exact duplicate rows; \
 (b) clean booking_channel with .str.strip().str.title(); \
 (c) convert appt_date with pd.to_datetime(..., errors="coerce") and booked_on with pd.to_datetime; \
 (d) drop rows where appt_date is missing after conversion or status is missing

In [8]:
df = df.drop_duplicates(ignore_index=True)
print(df.shape[0])

1000


In [9]:
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print(len(df))

1000


In [10]:
df['appt_date'] = pd.to_datetime(df['appt_date'], errors="coerce")
df['booked_on'] = pd.to_datetime(df['booked_on'])
print(df.shape[0])

1000


In [11]:
df = df.dropna(subset=["appt_date", "status"])
print(df.shape[0])

990


(a) Treat any age below 1 or above 100 as invalid and set it to missing.\
(b) Fill missing age with the median (then .round().astype(int)) and missing distance_km with its median. \
(c) For distance_km and fee_inr, compute Q1, Q3 and IQR with .quantile, and cap values at Q3 + 1.5×IQR and Q1 − 1.5×IQR with np.clip.\
(d) Round distance_km to 1 decimal and fee_inr to whole rupees (.round().astype(int)). Print how many values you capped in each column.

In [12]:
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan
df['age'] = df['age'].fillna(
    df['age'].median()
).round().astype(int)

df['distance_km'] = df['distance_km'].fillna(
    df['distance_km'].median()
)

In [13]:
q1_dist = df['distance_km'].quantile(0.25)
q3_dist = df['distance_km'].quantile(0.75)

IQR_dist = q3_dist - q1_dist

lower_dist = q1_dist - 1.5 * IQR_dist
upper_dist = q3_dist + 1.5 * IQR_dist

df['distance_km'] = np.clip(
    df['distance_km'], lower_dist, upper_dist
)

In [14]:
q1_fee_inr = df['fee_inr'].quantile(0.25)
q3_fee_inr = df['fee_inr'].quantile(0.75)

IQR_fee_inr = q3_fee_inr - q1_fee_inr

lower_fee_inr = q1_fee_inr - 1.5 * IQR_fee_inr
upper_fee_inr = q3_fee_inr + 1.5 * IQR_fee_inr

df['fee_inr'] = np.clip(
    df['fee_inr'], lower_fee_inr, upper_fee_inr
)

In [15]:
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

Task 1.4
1. Create lead_days = days between booked_on and appt_date (use .dt.days)
2. no_show = 1 if status is "No-Show" else 0.
3. take X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy()  without loops, compute: \
  (a) the mean of each column (axis=0);\
  (b) the number and share of rows with lead_days > 14 using a boolean mask (mask.sum() counts the True values and mask.mean() gives the share); \
  (c) z-scores of the distance_km column using (x - np.mean(x)) / np.std(x) and the count of rows with |z| > 2.

In [16]:
df['booked_on'] = pd.to_datetime(df['booked_on'])
df['appt_date'] = pd.to_datetime(df['appt_date'])
df['lead_days'] = (
    df['appt_date'] - df['booked_on']
).dt.days

In [17]:
df['no_show'] = (df['status'] == 'No-Show').astype(int)

In [18]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
x1 = np.mean(X, axis=0)
print(x1, 'x1')
lead_days = X[:, 3]
mask = lead_days > 14
count = mask.sum()
share = mask.mean()

x = X[:, 1]
z_score = (x - np.mean(x))/np.std(x)
z_score_count = (np.abs(z_score) > 2).sum()
print(z_score_count)

[ 48.17373737   9.50090909 635.71414141  11.81919192] x1
53


Task 1.5\
For each of the 8 clinics call the free Open-Meteo API with \
  requests.get(\
    "https://api.open-meteo.com/v1/forecast",\
    params={"latitude": ..., "longitude": ..., "hourly": "temperature_2m", "forecast_days": 1}\
  ). \
  Check status_code == 200 before using the reply, read the JSON, take the first value of ["hourly"]["temperature_2m"] (the midnight-UTC reading, a stand-in rather than the current weather) as temp_c, and save clinics_enriched.csv (the original columns plus temp_c; df.to_csv("file.csv", index=False) writes a DataFrame to CSV). \
  No API key is needed. \
  If the API fails: pass timeout=10 to requests.get, wrap the call in try/except requests.exceptions.RequestException, retry that clinic once, and if it still fails (or the status is not 200, or hourly is missing) store None as temp_c, print the clinic's name and the reason, and carry on. Never type in a made-up temperature; in load_data.sql a missing value is written as NULL (unquoted).

In [19]:
clinic_df = pd.read_csv('../data/jbFzPdd2gB2Ex2TY.csv')
clinic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   clinic_id    8 non-null      str    
 1   clinic_name  8 non-null      str    
 2   city         8 non-null      str    
 3   tier         8 non-null      str    
 4   latitude     8 non-null      float64
 5   longitude    8 non-null      float64
dtypes: float64(2), str(4)
memory usage: 516.0 bytes


In [20]:
def fetch_clinic_loc_temp(lat, long):
    try:
        api_request = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": long,
                "hourly": "temperature_2m",
                "forecast_days": 1
            }
        )

        if api_request.status_code == 200:
            response = api_request.json()
            print(api_request.json(), 'response')
            hourly = response.get('hourly', {})
            if hourly:
                return {
                    'success': True,
                    'temperature_2m': hourly.get('temperature_2m', [])[0]
                }

        return {
            'success': False,
            'reason': 'API Request was not successful'
        }
    except Exception as e:
        print(e)
        return {
            'success': False,
            'reason': str(e)
        }

In [21]:
temp_c_list = []
for index, row in clinic_df.iterrows():
    loc_temp_data = fetch_clinic_loc_temp(
        lat=row['latitude'], long=row['longitude'])
    if loc_temp_data.get('success'):
        temp_c_list.append(loc_temp_data.get('temperature_2m'))
    else:
        print(row['clinic_name'], loc_temp_data.get('reason'))
        temp_c_list.append(None)
clinic_df['temp_c'] = temp_c_list
print(clinic_df['temp_c'])

{'latitude': 18.523726, 'longitude': 73.78061, 'generationtime_ms': 0.021338462829589844, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 584.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C'}, 'hourly': {'time': ['2026-11-23T00:00', '2026-11-23T01:00', '2026-11-23T02:00', '2026-11-23T03:00', '2026-11-23T04:00', '2026-11-23T05:00', '2026-11-23T06:00', '2026-11-23T07:00', '2026-11-23T08:00', '2026-11-23T09:00', '2026-11-23T10:00', '2026-11-23T11:00', '2026-11-23T12:00', '2026-11-23T13:00', '2026-11-23T14:00', '2026-11-23T15:00', '2026-11-23T16:00', '2026-11-23T17:00', '2026-11-23T18:00', '2026-11-23T19:00', '2026-11-23T20:00', '2026-11-23T21:00', '2026-11-23T22:00', '2026-11-23T23:00'], 'temperature_2m': [18.3, 17.9, 19.2, 22.4, 25.2, 27.3, 29.1, 29.8, 30.2, 29.7, 29.6, 29.5, 28.4, 28.0, 26.4, 24.5, 24.1, 23.7, 23.3, 22.7, 22.0, 21.0, 20.4, 19.7]}} response
{'latitude': 18.523726, 'longitude': 73.95691, 'generationtime_ms': 0.01597404

In [22]:
clinic_df.to_csv("../data/clinics_enriched.csv", index=False)

Clinic File
1. INSERT INTO clinics VALUES (...) per clinic from clinics_enriched.csv with for loop and f-strings

Appointment File
1. Sort the cleaned data by appointment_id, format both date columns as YYYY-MM-DD text (.dt.strftime("%Y-%m-%d"))
2. Save appointments_clean.csv with exactly these 16 columns, in this order: appointment_id, patient_id, clinic_id, booked_on, appt_date, age, gender, distance_km, fee_inr, booking_channel, visit_type, reminder_sent, prior_no_shows, status, lead_days, no_show.
3. With a for loop and f-strings, write load_data.sql containing one statement per row, in this pattern (text in single quotes, numbers unquoted): INSERT INTO appointments VALUES ('A0001','P196','C05','2025-01-22',...,0);
4. Before exporting, check and print that appointment_id is unique and that none of the 16 columns has a missing value.

In [23]:
queries = []

In [24]:
cleaned_clinic_df = pd.read_csv('../data/clinics_enriched.csv')
queries = []
for index, row in cleaned_clinic_df.iterrows():
    query = (
        "INSERT INTO clinics (clinic_id, clinic_name, city, tier, "
        "latitude, longitude, temp_c)\n"
        f"VALUES ('{row['clinic_id']}', '{row['clinic_name']}', "
        f"'{row['city']}', '{row['tier']}', {row['latitude']}, "
        f"{row['longitude']}, {row['temp_c']});\n"
    )
    queries.append(query)

In [25]:
df = df.sort_values(by='appointment_id')
df['booked_on'] = df['booked_on'].dt.strftime('%Y-%m-%d')
df['appt_date'] = df['appt_date'].dt.strftime('%Y-%m-%d')

print('Duplicated Appointment Id count: \n',
      df['appointment_id'].duplicated().sum())
print("Missing values:\n", df.isnull().sum())

df.to_csv("../data/appointments_clean.csv", index=False)

Duplicated Appointment Id count: 
 0
Missing values:
 appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64


In [26]:
cleaned_appointments_df = pd.read_csv('../data/appointments_clean.csv')
for index, row in cleaned_appointments_df.iterrows():
    query = (
        "INSERT INTO appointments "
        "(appointment_id, patient_id, clinic_id, booked_on, appt_date, "
        "age, gender, distance_km, fee_inr, booking_channel, visit_type, "
        "reminder_sent, prior_no_shows, status, lead_days, no_show)\n"
        f"VALUES ('{row['appointment_id']}', '{row['patient_id']}', "
        f"'{row['clinic_id']}', '{row['booked_on']}', "
        f"'{row['appt_date']}', {row['age']}, '{row['gender']}', "
        f"{row['distance_km']}, {row['fee_inr']}, "
        f"'{row['booking_channel']}', '{row['visit_type']}', "
        f"'{row['reminder_sent']}', {row['prior_no_shows']}, "
        f"'{row['status']}', {row['lead_days']}, {row['no_show']});\n"
    )
    queries.append(query)

In [27]:
with open("../data/load_data.sql", "w", encoding='utf-8') as file:
    file.write("\n".join(queries))